### We assess the proportion of the genome that correspond to an overlap between two genes encoded on complementary strands

In [1]:
import pandas as pd
import pyranges as pr
from scipy.stats import ttest_ind
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
%reload_ext memory_profiler
import pickle

### **Functions def**

In [2]:
def load_whole_gtf(gtf_file):
    columns = ["Chromosome", "Source", "Feature", "Start", "End", "Score", "Strand", "Frame", "Attribute"]
    gtf = pd.read_csv(gtf_file, sep="\t", comment="#", names=columns)
    gtf["Chromosome"] = gtf["Chromosome"].astype(str)
    return gtf[["Chromosome", "Feature","Start", "End", "Strand", "Attribute"]]

In [3]:
def load_motifs(motifs_file):
    columns = ["Chromosome", "Start", "End", "Motif_name", "Strand", "Motif_Matched"]
    motifs = pd.read_csv(motifs_file, sep="\t", names=columns)
    motifs["second_idx"] = range(1, len(motifs) + 1)
    motifs["Chromosome"] = motifs["Chromosome"].astype(str)
    return motifs

In [4]:
# Scaffolds decrease confidence, we filter them out
def filter_chr(df, annotation_type):
    if annotation_type == 'Ensembl':
        resolved_chr = [f'{i}' for i in range(1,25+1)]
        resolved_chr.append('M')
    elif annotation_type == 'Lawson':
        resolved_chr = [f'chr{i}' for i in range(1,25+1)]
        resolved_chr.append('chrM')
    else:
        raise ErrorName('Provide a valid annotation')
        
    return df[df["Chromosome"].isin(resolved_chr)]

In [5]:
def get_percentage(numerator,denominator):
    return 100 * float(numerator) / float(denominator)

### **Files Loading and parameters**

In [6]:
level = 'gene' #'transcript'
annotation_type = 'Lawson' #'Ensembl'
case = 'case1'

In [7]:
if annotation_type == 'Ensembl':
    annotation_file = "../Reference/Ensembl/Danio_rerio.GRCz11.113.filtered.gtf"
    motifs_file = "../DCM_motifs_in_Genome/Danio_rerio/Ensembl/dcm_motifs_zero_based.bed"
    genome_length = 1373471384
elif annotation_type == 'Lawson':
    annotation_file = "/path_to_Reference/LawsonData/V4.3.2.validated.gtf"
    motifs_file = "../DCM_motifs_in_Genome/Danio_rerio/Lawson/dcm_motifs_zero_based.bed"
    genome_length = 1345101833
else:
    raise ErrorName('Provide a valid annotation')

### **Core code**

In [8]:
annotation = filter_chr(load_whole_gtf(annotation_file), annotation_type)

motifs_ = filter_chr(load_motifs(motifs_file),annotation_type)

In [9]:
features = annotation[annotation['Feature']==level]

In [10]:
features_forward = features[features['Strand']=='+']
features_reverse = features[features['Strand']=='-']

features_forward = pr.PyRanges(features_forward)
features_reverse = pr.PyRanges(features_reverse)

In [11]:
overlaps = features_forward.intersect(features_reverse, strandedness=False)

41473

**What is the proportion of DCM motifs falling inside these overlapping intervals?**

In [12]:
if case=='case1':
    strandedness_=False
elif case=="case2":
    strandedness_='same'
else:
    raise NameError('Provide a case number')

In [13]:
motifs = pr.PyRanges(motifs_)
inclusion_annotation = motifs.join(overlaps, strandedness=strandedness_, how='containment',report_overlap=True)
inclusion_annotation = inclusion_annotation.df[inclusion_annotation.df["Overlap"]>=5]

In [19]:
unique_motifs_in_overlaps = inclusion_annotation.drop_duplicates(subset=['second_idx'])
unique_motifs_in_overlaps.head(2)

,Chromosome,Start,End,Motif_name,Strand,Motif_Matched,second_idx,Feature,Start_b,End_b,Strand_b,Attribute,Overlap
0,chr1,11951,11956,chr1_11951_11956_for,+,CCTGG,29,gene,11822,12027,+,"gene_id ""LL0000005476""; gene_name ""cep97"";",5
1,chr1,109805,109810,chr1_109805_109810_for,+,CCAGG,228,gene,109798,110804,+,"gene_id ""LL0000009921""; gene_name ""F7 (1 of ma...",5
2,chr1,110707,110712,chr1_110707_110712_for,+,CCTGG,229,gene,109798,110804,+,"gene_id ""LL0000009921""; gene_name ""F7 (1 of ma...",5
3,chr1,160954,160959,chr1_160954_160959_for,+,CCAGG,303,gene,160302,168647,+,"gene_id ""LL0000008435""; gene_name ""cul4a"";",5
4,chr1,161194,161199,chr1_161194_161199_for,+,CCTGG,306,gene,160302,168647,+,"gene_id ""LL0000008435""; gene_name ""cul4a"";",5


In [20]:
#Consider filtered motifs, i.e. motifs found in several overlaps are removed (drop duplicates) 
total_motifs = len(motifs)
percent_motifs_in_overlaps = round(get_percentage(len(unique_motifs_in_overlaps),total_motifs),2)
print(f"{percent_motifs_in_overlaps}% of the motifs are found inside overlaps in {case}")

16.56% of the motifs are found inside overlaps in case1


**Lengths and proportions**

In [22]:
length_overlaps = overlaps.merge(strand=True).length #same results for strand=True or strand=False

In [23]:
#enrichment looking at genome length
features = pr.PyRanges(features)

if level == 'gene':
    # Total gene length = individual genes lengths - overlapping parts merging both strands
    genes_lengths = features.merge(strand=False).length
    #compare to whole genome:
    percent_overlap_genome = round(get_percentage(length_overlaps,genome_length),2)
    #compare to all genes:
    percent_overlap_genes = round(get_percentage(length_overlaps,genes_lengths),2)

    print(f"Percentage of total genome that correspond to an overlap of genes encoded on distinct strands: {percent_overlap_genome}% \nPercent of overlap compared to total genes lengths: {percent_overlap_genes}%")

if level == 'transcript':
    transcripts_lengths = features.merge(strand=True).length
    percent_overlap_transcripts = round(get_percentage(length_overlaps,transcripts_lengths),2)

    print(f"Proportion of the sum of all transcripts overlapped by transcripts encoded on distinct strands that are overlapping: {percent_overlap_transcripts}")

Percentage of total genome that correspond to an overlap of genes encoded on distinct strands: 15.28% 
Percent of overlap compared to total genes lengths: 19.59%
